# 01. 채널 목록 수집 (Channel Collection)

vling.net 필터링 검색 결과에서 유튜브 채널명과 채널 ID를 수집해 `YT_channelsList_v1.csv`로 저장합니다.

## 수집 흐름

```
vling.net 검색 URL
    ↓ (Playwright 브라우저 자동화)
채널 카드 클릭 → YouTube 채널 URL 추출
    ↓
YouTube URL에서 channel_id 파싱
    또는 핸들(@)이면 YouTube API로 조회
    ↓
YT_channelsList_v1.csv 저장
```

> API 키는 `.env` 파일에 저장합니다. `.env`는 `.gitignore`에 포함되어 있어 레포에 올라가지 않습니다.

---
## 패키지 설치

Playwright는 처음 한 번만 설치하면 됩니다.

In [ ]:
%pip install playwright nest_asyncio --quiet
import subprocess
subprocess.run(["playwright", "install", "chromium"], check=True)

---
## API 키 설정

프로젝트 루트의 `.env` 파일에 아래 형식으로 API 키를 저장하세요:

```
YOUTUBE_API_KEY=여기에_발급받은_키_입력
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError(".env 파일에 YOUTUBE_API_KEY가 설정되지 않았습니다.")
print("API 키 로드 완료")

---
## 라이브러리 임포트

In [ ]:
import asyncio
import re
import os
import nest_asyncio
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from googleapiclient.discovery import build
import requests

# vling.net → YT_channelsList_v1.csv
CHANNELS_LIST_CSV = Path("YT_channelsList_v1.csv")

---
## 채널 ID 조회 유틸

`@핸들명`으로 YouTube 채널 ID를 조회합니다. vling에서 `/@handle` 형식 URL만 얻었을 때 사용합니다.

In [ ]:
def get_channel_id_from_handle(api_key: str, handle: str) -> str | None:
    """
    YouTube 채널 핸들(@포함 또는 미포함)로 channel_id를 조회합니다.
    예: get_channel_id_from_handle(API_KEY, "@재활의학과탑팀")
    """
    youtube = build("youtube", "v3", developerKey=api_key)
    clean_handle = handle.replace("@", "")
    response = youtube.channels().list(
        part="id",
        forHandle=clean_handle,
    ).execute()

    if response.get("items"):
        return response["items"][0]["id"]
    return None


# ── 사용 예시 ──────────────────────────────────────────────────────────────
# ch_id = get_channel_id_from_handle(API_KEY, "@재활의학과탑팀")
# print(ch_id)  # → UCWW--vYvkR604lbGg92J2_A
print("get_channel_id_from_handle 함수 정의 완료")

---
## vling.net 크롤러

### vling.net 크롤러

각 채널 카드를 클릭해 vling 채널 페이지로 들어가면, 유튜브 채널 링크가 있습니다.  
그 링크에서 `channel_id` 또는 `@handle`을 파싱합니다.

In [ ]:
import asyncio
import re
import nest_asyncio
import pandas as pd
from pathlib import Path
from playwright.async_api import async_playwright

nest_asyncio.apply()  # Jupyter 내 asyncio 중첩 허용


async def scrape_vling(
    search_url: str,
    age_label: str,                  # YT_channelsList_v1.csv의 'age' 컬럼값
    max_channels: int = 100,
    headless: bool = True,
) -> list[dict]:
    """
    vling.net 검색 결과에서 채널명 + YouTube 채널 URL을 수집합니다.
    각 채널 카드를 클릭 → vling 채널 상세 페이지 → YouTube 링크 파싱
    """
    results = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        ctx = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            )
        )
        page = await ctx.new_page()

        print(f"페이지 로딩 중...")
        await page.goto(search_url, wait_until="networkidle", timeout=30_000)
        await page.wait_for_timeout(2000)

        # 스크롤 다운 → lazy-load 채널 카드 전부 렌더링
        prev_count = 0
        for _ in range(20):
            await page.evaluate("window.scrollBy(0, window.innerHeight * 3)")
            await page.wait_for_timeout(1200)
            cards = await page.query_selector_all("a[href*='/channel/']")
            if len(cards) >= max_channels or len(cards) == prev_count:
                break
            prev_count = len(cards)

        cards = await page.query_selector_all("a[href*='/channel/']")
        print(f"채널 카드 발견: {len(cards)}개")

        for i, card in enumerate(cards[:max_channels]):
            channel_name = (await card.inner_text()).strip().split("\n")[0]
            vling_href   = await card.get_attribute("href")
            if not vling_href:
                continue

            vling_url = f"https://vling.net{vling_href}" if vling_href.startswith("/") else vling_href

            # 채널 상세 페이지에서 YouTube URL 추출
            detail = await ctx.new_page()
            yt_url, channel_id, handle = "", "", ""
            try:
                await detail.goto(vling_url, wait_until="networkidle", timeout=20_000)
                await detail.wait_for_timeout(1500)

                # YouTube 링크 href 찾기
                yt_link = await detail.query_selector("a[href*='youtube.com/channel/'], a[href*='youtube.com/@']")
                if yt_link:
                    yt_url = await yt_link.get_attribute("href")

                    # /channel/UC... 형식
                    m = re.search(r"youtube\.com/channel/(UC[\w-]+)", yt_url)
                    if m:
                        channel_id = m.group(1)

                    # /@handle 형식 → API로 나중에 변환
                    m2 = re.search(r"youtube\.com/@([\w.-]+)", yt_url)
                    if m2:
                        handle = "@" + m2.group(1)

            except Exception as e:
                print(f"  [{i+1}] {channel_name}: 상세 페이지 오류 → {e}")
            finally:
                await detail.close()

            results.append({
                "channel_name": channel_name,
                "channel_id":   channel_id,
                "handle":       handle,
                "youtube_url":  yt_url,
                "age":          age_label,
            })
            status = channel_id or handle or "ID 미확인"
            print(f"  [{i+1:>3}] {channel_name[:25]:<25} → {status}")

        await browser.close()

    return results


print("scrape_vling 함수 정의 완료")

---
## handle → channel_id 변환

### handle → channel_id 변환

vling 상세 페이지에 `/@handle` 형식 URL만 있는 채널은 YouTube API로 `channel_id`를 조회합니다.

In [ ]:
def resolve_channel_ids(rows: list[dict], api_key: str) -> list[dict]:
    """handle만 있고 channel_id가 없는 행을 YouTube API로 보완합니다."""
    for row in rows:
        if row["channel_id"] or not row["handle"]:
            continue
        ch_id = get_channel_id_from_handle(api_key, row["handle"])
        if ch_id:
            row["channel_id"] = ch_id
            print(f"  resolved: {row['handle']} → {ch_id}")
        else:
            print(f"  미확인: {row['handle']}")
    return rows


print("resolve_channel_ids 함수 정의 완료")

---
## 실행

### 실행

`VLING_URLS`에 수집하고 싶은 카테고리별 URL과 연령대 레이블을 입력하세요.  
현재 예시는 스크린샷의 **헬스 / 성인** 카테고리 URL입니다.

In [ ]:
# ── 수집 대상 URL 목록 ────────────────────────────────────────────────────
# (url, age_label) 형태로 추가
VLING_URLS = [
    (
        "https://vling.net/search?sort=dailyAverageViewCount&na=KR"
        "&category=HEALTH&subscriber=10K_100K%2C100K_500K%2C500K_1M%2C1M_over"
        "&upload=1&isShorts=false",
        "adult",       # 연령대 레이블 (pipeline CSV의 'age' 컬럼)
    ),
    # 다른 카테고리 추가 예시:
    # ("https://vling.net/search?...", "teen"),
    # ("https://vling.net/search?...", "child"),
]

CHANNELS_LIST_CSV = Path("YT_channelsList_v1.csv")
MAX_CHANNELS      = 100   # URL당 최대 수집 채널 수

# ── 실행 ─────────────────────────────────────────────────────────────────
all_rows = []
for url, label in VLING_URLS:
    print(f"\n[{label}] 수집 시작: {url[:60]}...")
    rows = asyncio.run(scrape_vling(url, age_label=label, max_channels=MAX_CHANNELS))
    all_rows.extend(rows)

# handle → channel_id 보완
print("\nhandle → channel_id 변환 중...")
all_rows = resolve_channel_ids(all_rows, API_KEY)

# 저장
df_new = pd.DataFrame(all_rows)
print(f"\n--- 수집 결과 ---")
print(f"총 채널 수      : {len(df_new)}")
print(f"channel_id 확보 : {df_new['channel_id'].astype(bool).sum()} / {len(df_new)}")
print(f"ID 미확인       : {(~df_new['channel_id'].astype(bool)).sum()}개")
df_new.head(10)

---
## 결과 저장

### 결과 저장

`channel_id`가 있는 채널만 `YT_channelsList_v1.csv`에 저장합니다.  
기존 파일이 있으면 중복 없이 합쳐서 저장합니다.

In [ ]:
save_cols = ["channel_name", "channel_id", "age"]

df_save = df_new[df_new["channel_id"].astype(bool)][save_cols].copy()

if CHANNELS_LIST_CSV.exists():
    df_exist = pd.read_csv(CHANNELS_LIST_CSV, encoding="utf-8-sig")
    df_save  = pd.concat([df_exist, df_save], ignore_index=True)
    df_save  = df_save.drop_duplicates(subset=["channel_id"])

df_save.to_csv(CHANNELS_LIST_CSV, index=False, encoding="utf-8-sig")

print(f"저장 완료: {len(df_save)}개 채널 → {CHANNELS_LIST_CSV}")
print(df_save.groupby("age").size().rename("채널 수").to_string())

# channel_id 없는 채널 확인
no_id = df_new[~df_new["channel_id"].astype(bool)][["channel_name", "handle", "youtube_url"]]
if not no_id.empty:
    print(f"\n⚠ channel_id 미확인 채널 ({len(no_id)}개) — 수동 확인 필요:")
    print(no_id.to_string(index=False))